[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day02-tokenization-bpe-from-scratch.ipynb)

# Day 2 — Tokenization: BPE from scratch

Today you implement byte-pair encoding in ~40 lines of pure Python, train it on Tiny Shakespeare, and compare it against a production tokenizer (tiktoken). Then you convert words → tokens → dollars.

Runs on CPU (no GPU needed). ~30 minutes. Run every cell top to bottom: **Runtime → Run all**.

## 0. Environment check

In [ ]:
import sys
print('python:', sys.version.split()[0])
# Expected: python: 3.10+ (any recent version works)

## 1. Install dependencies

`tiktoken` (OpenAI's production BPE tokenizer) and `matplotlib` for the comparison plot.

In [ ]:
%pip install --quiet tiktoken matplotlib
# Expected: a short install log (or "already satisfied").

## 2. Download Tiny Shakespeare (~1.1 MB)

In [ ]:
import urllib.request, os
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(url, 'input.txt')
text = open('input.txt').read()
print('chars:', len(text))
print('first 120 chars:', repr(text[:120]))
# Expected: chars: 1115394

## 3. The BPE trainer: `get_stats` + `merge`, 50 merges

Start from raw bytes (vocab = 256). Each round: count every adjacent pair, merge the most frequent one into a new token.

In [ ]:
from collections import Counter

def get_stats(ids):
    """Count occurrences of every adjacent pair."""
    return Counter(zip(ids, ids[1:]))

def merge(ids, pair, idx):
    """Rewrite the sequence, replacing every `pair` with the new token `idx`."""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

ids = list(text.encode('utf-8'))
vocab = {i: bytes([i]) for i in range(256)}  # 256 byte tokens, always present
merges = {}  # (a, b) -> new token id, in merge order

for n in range(50):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)      # most frequent adjacent pair
    idx = 256 + n
    ids = merge(ids, pair, idx)
    merges[pair] = idx
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    if n in (0, 9, 49):
        print(f'merge #{n+1}: {pair} -> {vocab[idx]!r}  (was {stats[pair]} occurrences)')

print('vocab size:', len(vocab))  # 256 + 50
# Expected (exact first merges depend on tie-breaking, yours will be close):
# merge #1: (101, 32) -> b'e '   <- 'e' followed by space is the most common pair
# merge #10 and #50 assemble common fragments like 'th', 'he', ' the'
# vocab size: 306

## 4. Encode and decode: the round-trip check

Apply merges greedily in learned order, then verify decode(encode(s)) == s.

In [ ]:
def encode(s):
    ids = list(s.encode('utf-8'))
    for pair, idx in merges.items():
        ids = merge(ids, pair, idx)
    return ids

def decode(ids):
    raw = b''.join(vocab[i] for i in ids)
    return raw.decode('utf-8', errors='replace')

s = 'To be, or not to be'
mine = encode(s)
print('my token ids:', mine)
print('my token count:', len(mine))
print('pieces:', [vocab[i] for i in mine])
assert decode(s and mine) == s, 'round-trip failed'
print('round-trip OK:', decode(mine) == s)
# Expected: my token count: 11  (50 merges is a toy; production does 100k)

## 5. Compare against production: tiktoken `cl100k_base` (GPT-4's tokenizer)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('cl100k_base')
prod = enc.encode(s)
print('tiktoken ids:', prod)
print('tiktoken count:', len(prod))
print('pieces:', [enc.decode([i]) for i in prod])
assert len(prod) == 7, 'expected exactly 7 tokens'
# Expected: tiktoken count: 7
# pieces: ['To', ' be', ',', ' or', ' not', ' to', ' be']

## 6. Compression shootout: characters per token

Higher is better. English prose should land near ~4 chars/token for a good tokenizer.

In [ ]:
sample = text[:10_000]
mine_cpt = len(sample) / len(encode(sample))
prod_cpt = len(sample) / len(enc.encode(sample))
print(f'my 50-merge BPE : {mine_cpt:.2f} chars/token')
print(f'tiktoken cl100k : {prod_cpt:.2f} chars/token')
print(f'char-level      : 1.00 chars/token (by definition)')
# Expected: my 50-merge BPE : 1.45 chars/token; tiktoken cl100k : 3.89 chars/token

## 7. Plot it: cumulative tokens vs characters

Slope = tokens per character. Shallower slope = better compression.

In [ ]:
import matplotlib.pyplot as plt

xs, my_ys, prod_ys = [], [], []
chunk = text[:10_000]
for n in range(500, 10_001, 500):
    piece = chunk[:n]
    xs.append(n)
    my_ys.append(len(encode(piece)))
    prod_ys.append(len(enc.encode(piece)))

plt.figure()
plt.plot(xs, my_ys, label='my 50-merge BPE')
plt.plot(xs, prod_ys, label='tiktoken cl100k_base')
plt.xlabel('characters')
plt.ylabel('tokens')
plt.title('Token count vs characters (Tiny Shakespeare)')
plt.legend()
plt.show()
# Expected: two straight lines; tiktoken's slope is shallower.

## 8. The money math: words → tokens → dollars

Rule of thumb: English ≈ 0.75 words/token. Pricing here: $2.50 / 1M input tokens.

In [ ]:
def cost_usd(words, price_per_mtok=2.50, words_per_token=0.75):
    tokens = words / words_per_token
    return tokens * price_per_mtok / 1e6

for w in (500, 2_000, 100_000):
    print(f'{w:>7,} words -> ~{w/0.75:>9,.0f} tokens -> ${cost_usd(w):.4f}')
print()
print('code/mixed text tokenizes ~2-3x worse: multiply the token column by 2-3')
# Expected:
#     500 words -> ~      667 tokens -> $0.0017
#   2,000 words -> ~    2,667 tokens -> $0.0067
# 100,000 words -> ~  133,333 tokens -> $0.3333

## 9. Record your numbers

Copy these into the "What to measure" table in today's PDF:

- your BPE's token count for "To be, or not to be"
- tiktoken's count (should be 7)
- both chars/token figures from §6
- the dollar cost of a 2,000-word prompt

**Stretch:** encode a line of Python and a non-English sentence with both tokenizers. Watch the chars/token gap widen for code — that gap is money on every request you will ever serve.

Tomorrow (Day 3): attention math by hand — softmax(QKᵀ/√d_k)V on a tiny 3-token example, and the O(n²) blowup, measured.